# NB03 — Hospital Charges & Medicare Payments from CMS Inpatient PUF

**Purpose:** Download the CMS Medicare Inpatient Hospitals Provider Utilization & Payment (PUF) file to get hospital-level charges and Medicare payments broken down by DRG. This data lets us see exactly how much hospitals charge vs. how much Medicare pays for each type of case — and where the biggest revenue gaps exist.

**What we're building:**
- Hospital × DRG level charge and payment data
- Aggregated hospital-level payment statistics (total charges, total payments, charge-to-payment ratio)
- DRG-level payment distributions to identify high-variance DRGs where documentation accuracy has the biggest financial impact

**Data Source:** [Medicare Inpatient Hospitals - By Provider and Service](https://data.cms.gov/provider-summary-by-type-of-service/medicare-inpatient-hospitals/medicare-inpatient-hospitals-by-provider-and-service)  
This is a direct CSV download from data.cms.gov. No API key needed.

**Output:** `data/outputs/nb03_payment_data/hospital_payment_data.csv`

---

### Domain Context: Hospital Charges vs. Medicare Payments

There's a crucial distinction in hospital finance:

- **Charges (Covered Charges):** The hospital's list price — what they bill to the insurer. Charges are often 3-5x what anyone actually pays. They're set by the hospital's chargemaster and have only a loose relationship to actual costs.
- **Payments (Medicare Allowed Amount):** What Medicare actually pays, based on the DRG weight × base rate × hospital adjustments. This is the real revenue.
- **Charge-to-Payment Ratio (CPR):** Charges ÷ Payments. A CPR of 3.5 means the hospital bills $3.50 for every $1.00 Medicare pays. Higher CPR hospitals may be "charging up" to compensate for under-documentation.

**Why this matters for CDI:** The PUF shows us payments by DRG, so we can calculate:
1. How much a hospital receives for each DRG
2. Which DRGs have the widest variation in payment (indicating potential documentation-driven differences)
3. The total Medicare revenue per hospital (to contextualize NB08's revenue gap estimates)

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import os
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('..').resolve().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'nb03_payment_data'

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')

Project root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap
Raw data dir: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw
Output dir:   /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb03_payment_data


## 2. Download the Medicare Inpatient PUF

### What is the Inpatient PUF?

The **Medicare Inpatient Hospitals - Provider Utilization and Payment Data** (commonly called the Inpatient PUF) contains information on services provided to Medicare fee-for-service beneficiaries by hospital inpatient providers. Each row represents a **hospital × DRG** combination and includes:

- **Provider ID (CCN):** Links to NB01 and NB02
- **DRG Definition:** The MS-DRG code and description
- **Total Discharges:** Number of Medicare discharges for this DRG at this hospital
- **Average Covered Charges:** Mean charges billed
- **Average Total Payments:** Mean Medicare payment received
- **Average Medicare Payments:** Mean Medicare portion only (excluding beneficiary copays)

The file covers about **3,100+ hospitals** across **750+ DRGs**, resulting in several hundred thousand rows. CMS suppresses cells with fewer than 11 discharges to protect patient privacy, so very small hospitals or rare DRGs won't appear.

The most recent release is typically Reporting Year 2024 or 2025 (using discharge year 2022 or 2023 data).

In [2]:
# ============================================================
# Download the Inpatient PUF
# ============================================================
# The PUF is available as a large CSV from data.cms.gov.
# File naming pattern: MUP_INP_RY{year}_P03_V10_DY{year-2}_PrvSvc.CSV
# Dataset UUID (verified from data.cms.gov Access API modal):
#   690ddc6c-2767-4618-b277-420ffb2bf27c

PUF_DATASET_UUID = '690ddc6c-2767-4618-b277-420ffb2bf27c'

# CSV bulk download (data-api endpoint)
PUF_CSV_URL = f'https://data.cms.gov/data-api/v1/dataset/{PUF_DATASET_UUID}/data.csv'

# JSON API (paginated fallback)
PUF_JSON_URL = f'https://data.cms.gov/data-api/v1/dataset/{PUF_DATASET_UUID}/data'

# Legacy direct file URLs (CMS changes these frequently)
PUF_LEGACY_URLS = [
    # RY 2025 (DY 2023)
    'https://data.cms.gov/sites/default/files/2025-01/MUP_INP_RY25_P03_V10_DY23_PrvSvc.CSV',
    'https://data.cms.gov/sites/default/files/2024-12/MUP_INP_RY25_P03_V10_DY23_PrvSvc.CSV',
    # RY 2024 (DY 2022)
    'https://data.cms.gov/sites/default/files/2024-05/MUP_INP_RY24_P03_V10_DY22_PrvSvc.CSV',
    'https://data.cms.gov/sites/default/files/2024-06/MUP_INP_RY24_P03_V10_DY22_PrvSvc.CSV',
    'https://data.cms.gov/sites/default/files/2024-04/MUP_INP_RY24_P03_V10_DY22_PrvSvc.CSV',
]

puf_raw_path = RAW_DIR / 'inpatient_puf.csv'

if puf_raw_path.exists() and puf_raw_path.stat().st_size > 1_000_000:
    print(f'PUF file already exists: {puf_raw_path}')
    print(f'File size: {puf_raw_path.stat().st_size / 1e6:.1f} MB')
else:
    downloaded = False
    
    # --- Attempt 1: CSV bulk download via data-api ---
    print('Attempt 1: CSV bulk download from data-api endpoint...')
    try:
        resp = requests.get(PUF_CSV_URL, timeout=300, stream=True)
        resp.raise_for_status()
        size = int(resp.headers.get('Content-Length', 0))
        print(f'  Response: HTTP {resp.status_code}, Content-Length: {size:,} bytes')
        if size > 1_000_000 or resp.headers.get('Content-Type','').startswith('text/csv'):
            with open(puf_raw_path, 'wb') as f:
                for chunk in resp.iter_content(65536):
                    f.write(chunk)
            actual_size = puf_raw_path.stat().st_size
            if actual_size > 1_000_000:
                print(f'  Downloaded {actual_size / 1e6:.1f} MB')
                downloaded = True
            else:
                print(f'  File too small ({actual_size} bytes) — removing')
                puf_raw_path.unlink()
    except Exception as e:
        print(f'  Failed: {e}')
    
    # --- Attempt 2: Legacy direct CSV URLs ---
    if not downloaded:
        print('\nAttempt 2: Legacy direct CSV URLs...')
        for url in PUF_LEGACY_URLS:
            print(f'  Trying: {url.split("/")[-1]}')
            try:
                resp = requests.get(url, timeout=300, stream=True)
                if resp.status_code == 200:
                    total_size = int(resp.headers.get('Content-Length', 0))
                    if total_size > 1_000_000:
                        with open(puf_raw_path, 'wb') as f:
                            for chunk in resp.iter_content(65536):
                                f.write(chunk)
                        print(f'  Downloaded {puf_raw_path.stat().st_size / 1e6:.1f} MB')
                        downloaded = True
                        break
                    else:
                        print(f'  File too small ({total_size} bytes)')
                else:
                    print(f'  HTTP {resp.status_code}')
            except Exception as e:
                print(f'  Error: {e}')
    
    # --- Attempt 3: JSON API with pagination ---
    if not downloaded:
        print('\nAttempt 3: JSON API with pagination (this is slow for large datasets)...')
        try:
            test = requests.get(f'{PUF_JSON_URL}?size=1', timeout=30)
            test.raise_for_status()
            print('  API responsive. Fetching all records in chunks of 5000...')
            
            all_records = []
            offset = 0
            page_size = 5000
            max_pages = 200
            
            for page in range(max_pages):
                url = f"{PUF_JSON_URL}?size={page_size}&offset={offset}"
                if page % 10 == 0:
                    print(f'  Page {page+1}: fetching records {offset:,} to {offset+page_size:,}...')
                
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                batch = r.json()
                
                if not batch:
                    break
                
                all_records.extend(batch)
                offset += page_size
                
                if len(batch) < page_size:
                    break
            
            print(f'  Total: {len(all_records):,} records')
            if len(all_records) > 1000:
                df_puf_api = pd.DataFrame(all_records)
                df_puf_api.to_csv(puf_raw_path, index=False)
                print(f'  Saved {puf_raw_path.stat().st_size / 1e6:.1f} MB')
                downloaded = True
        except Exception as e:
            print(f'  JSON API failed: {e}')
    
    if not downloaded:
        print('\n' + '=' * 60)
        print('MANUAL DOWNLOAD REQUIRED')
        print('=' * 60)
        print('Please download the Inpatient PUF manually:')
        print('1. Go to: https://data.cms.gov/provider-summary-by-type-of-service/')
        print('   medicare-inpatient-hospitals/medicare-inpatient-hospitals-by-provider-and-service')
        print('2. Click "Download" → "Latest Dataset Only" → "Download Files"')
        print(f'3. Save CSV to: {puf_raw_path}')
        print('Then re-run this cell.')

PUF file already exists: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/inpatient_puf.csv
File size: 39.2 MB


## 3. Load & Explore the PUF Data

The PUF has one row per hospital × DRG combination. Let's understand the structure before we start cleaning.

In [3]:
# ============================================================
# Load the raw PUF file
# ============================================================
# CMS files sometimes use Latin-1 (cp1252) encoding instead of UTF-8.
# Try UTF-8 first, fall back to Latin-1.

try:
    df_puf_raw = pd.read_csv(puf_raw_path, dtype=str, low_memory=False, encoding='utf-8')
except UnicodeDecodeError:
    print('UTF-8 failed — trying Latin-1 encoding...')
    df_puf_raw = pd.read_csv(puf_raw_path, dtype=str, low_memory=False, encoding='latin-1')

print(f'Raw PUF file: {len(df_puf_raw):,} rows')
print(f'Columns ({len(df_puf_raw.columns)}):')
for i, col in enumerate(df_puf_raw.columns):
    sample = df_puf_raw[col].dropna().iloc[0] if df_puf_raw[col].notna().any() else 'N/A'
    print(f'  {i:3d}. {col:50s} sample: {str(sample)[:40]}')

print(f'\nFirst 3 rows:')
df_puf_raw.head(3)

UTF-8 failed — trying Latin-1 encoding...


Raw PUF file: 146,427 rows
Columns (15):
    0. Rndrng_Prvdr_CCN                                   sample: 010001
    1. Rndrng_Prvdr_Org_Name                              sample: Southeast Health Medical Center
    2. Rndrng_Prvdr_City                                  sample: Dothan
    3. Rndrng_Prvdr_St                                    sample: 1108 Ross Clark Circle
    4. Rndrng_Prvdr_State_FIPS                            sample: 01
    5. Rndrng_Prvdr_Zip5                                  sample: 36301
    6. Rndrng_Prvdr_State_Abrvtn                          sample: AL
    7. Rndrng_Prvdr_RUCA                                  sample: 2
    8. Rndrng_Prvdr_RUCA_Desc                             sample: Metropolitan area high commuting: primar
    9. DRG_Cd                                             sample: 003
   10. DRG_Desc                                           sample: ECMO OR TRACHEOSTOMY WITH MV >96 HOURS O
   11. Tot_Dschrgs                                        sample

,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt
0,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,14,663764.35714,120219.92857,115544.14286
1,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,023,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,26,180980.88462,37321.038462,35261.807692
2,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,024,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,12,105824.33333,26936.666667,25048.916667


In [4]:
# Explore key field values
print('Unique providers:', df_puf_raw.iloc[:, 0].nunique())
print('Unique DRGs:', df_puf_raw.iloc[:, -4].nunique() if len(df_puf_raw.columns) > 4 else 'unknown')
print(f'\nSample rows from different hospitals:')
df_puf_raw.sample(5, random_state=42)

Unique providers: 2945
Unique DRGs: 655

Sample rows from different hospitals:


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt
980,010039,Huntsville Hospital,Huntsville,101 Sivley Rd,01,35801,AL,1,Metropolitan area core: primary flow within an...,602,CELLULITIS WITH MCC,25,61561.48,9919.64,8826.28
36234,100315,Viera Hospital,Melbourne,8745 N Wickham Rd,12,32940,FL,1,Metropolitan area core: primary flow within an...,948,SIGNS AND SYMPTOMS WITHOUT MCC,12,31716,4852.6666667,3652.6666667
36588,110001,Hamilton Medical Center,Dalton,1200 Memorial Drive,13,30720,GA,1,Metropolitan area core: primary flow within an...,521,HIP REPLACEMENT WITH PRINCIPAL DIAGNOSIS OF HI...,15,126405.8,23289.466667,21010.8
61626,210043,Umd Baltimore Washington Medical Center,Glen Burnie,301 Hospital Drive,24,21061,MD,1,Metropolitan area core: primary flow within an...,035,CAROTID ARTERY STENT PROCEDURES WITH CC,12,31060.083333,27872.916667,26276.583333
85278,310050,Saint Clare's Hospital,Denville,25 Pocono Road,34,07834,NJ,1,Metropolitan area core: primary flow within an...,074,CRANIAL AND PERIPHERAL NERVE DISORDERS WITHOUT...,33,29572.727273,9541.6060606,8260.6363636


## 4. Extract & Rename Key Columns

### PUF column mapping

The PUF uses descriptive column names that vary slightly by release year. Common patterns:

| Our Field | Common PUF Column Names |
|-----------|------------------------|
| `ccn` | `Rndrng_Prvdr_CCN`, `Provider Id`, `PRVDR_ID` |
| `hospital_name` | `Rndrng_Prvdr_Org_Name`, `Provider Name` |
| `drg_code` | `DRG_Cd`, `DRG Definition`, `MS_DRG_CD` |
| `drg_description` | `DRG_Desc`, `DRG Definition` |
| `discharges` | `Tot_Dschrgs`, `Total Discharges` |
| `avg_covered_charges` | `Avg_Submtd_Cvrd_Chrg`, `Average Covered Charges` |
| `avg_total_payment` | `Avg_Tot_Pymt_Amt`, `Average Total Payments` |
| `avg_medicare_payment` | `Avg_Mdcr_Pymt_Amt`, `Average Medicare Payments` |

In [5]:
# ============================================================
# Map PUF columns to standardized names
# ============================================================

def find_col(df, keywords, exclude=None):
    """Find a column name containing any of the keywords."""
    exclude = exclude or []
    for col in df.columns:
        col_upper = col.upper().replace(' ', '_')
        if any(kw in col_upper for kw in keywords):
            if not any(ex in col_upper for ex in exclude):
                return col
    return None

col_map = {
    'ccn': find_col(df_puf_raw, ['CCN', 'PRVDR_ID', 'PROVIDER_ID', 'RNDRNG_PRVDR_CCN']),
    'hospital_name': find_col(df_puf_raw, ['ORG_NAME', 'PROVIDER_NAME', 'PRVDR_ORG', 'RNDRNG_PRVDR_ORG']),
    'state': find_col(df_puf_raw, ['STATE', 'ST_'], exclude=['FIPS']),
    'drg_code': find_col(df_puf_raw, ['DRG_CD', 'DRG_CODE', 'MS_DRG']),
    'drg_description': find_col(df_puf_raw, ['DRG_DESC', 'DRG_DEFINITION', 'DRG_DEF']),
    'discharges': find_col(df_puf_raw, ['DSCHRG', 'DISCHARGE', 'TOT_DSCHRGS']),
    'avg_covered_charges': find_col(df_puf_raw, ['CVRD_CHRG', 'COVERED_CHARGE', 'SUBMTD_CVRD']),
    'avg_total_payment': find_col(df_puf_raw, ['TOT_PYMT', 'TOTAL_PAYMENT', 'TOT_PMT']),
    'avg_medicare_payment': find_col(df_puf_raw, ['MDCR_PYMT', 'MEDICARE_PAYMENT', 'MDCR_PMT']),
}

print('Column mapping:')
for field, col in col_map.items():
    status = '✓' if col else '✗ NOT FOUND'
    print(f'  {field:25s} → {col or status}')

# Build clean dataframe
rename_map = {v: k for k, v in col_map.items() if v is not None}
df_puf = df_puf_raw[list(rename_map.keys())].rename(columns=rename_map).copy()

print(f'\nCleaned PUF: {len(df_puf):,} rows')

Column mapping:
  ccn                       → Rndrng_Prvdr_CCN
  hospital_name             → Rndrng_Prvdr_Org_Name
  state                     → Rndrng_Prvdr_State_Abrvtn
  drg_code                  → DRG_Cd
  drg_description           → DRG_Desc
  discharges                → Tot_Dschrgs
  avg_covered_charges       → Avg_Submtd_Cvrd_Chrg
  avg_total_payment         → Avg_Tot_Pymt_Amt
  avg_medicare_payment      → Avg_Mdcr_Pymt_Amt



Cleaned PUF: 146,427 rows


In [6]:
# ============================================================
# Convert numeric columns & clean CCN
# ============================================================

numeric_cols = ['discharges', 'avg_covered_charges', 'avg_total_payment', 'avg_medicare_payment']
for col in numeric_cols:
    if col in df_puf.columns:
        # Remove $ signs, commas, and convert to float
        df_puf[col] = (
            df_puf[col].astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df_puf[col] = pd.to_numeric(df_puf[col], errors='coerce')

# Clean CCN
if 'ccn' in df_puf.columns:
    df_puf['ccn'] = df_puf['ccn'].astype(str).str.strip().str.zfill(6)

# Extract DRG number from description if drg_code wasn't found separately
if 'drg_code' not in df_puf.columns or df_puf['drg_code'].isna().all():
    if 'drg_description' in df_puf.columns:
        # DRG definitions are often formatted as "XXX - Description"
        df_puf['drg_code'] = df_puf['drg_description'].str.extract(r'(\d{3})', expand=False)
        print('Extracted DRG codes from description field.')

# Convert DRG code to integer
if 'drg_code' in df_puf.columns:
    df_puf['drg_code'] = pd.to_numeric(df_puf['drg_code'], errors='coerce')

print('Data types after conversion:')
print(df_puf.dtypes.to_string())
print(f'\nSample rows:')
df_puf.head(5)

Data types after conversion:
ccn                      object
hospital_name            object
state                    object
drg_code                  int64
drg_description          object
discharges                int64
avg_covered_charges     float64
avg_total_payment       float64
avg_medicare_payment    float64

Sample rows:


,ccn,hospital_name,state,drg_code,drg_description,discharges,avg_covered_charges,avg_total_payment,avg_medicare_payment
0,010001,Southeast Health Medical Center,AL,3,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,14,663764.35714,120219.928570,115544.142860
1,010001,Southeast Health Medical Center,AL,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,26,180980.88462,37321.038462,35261.807692
2,010001,Southeast Health Medical Center,AL,24,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,12,105824.33333,26936.666667,25048.916667
3,010001,Southeast Health Medical Center,AL,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,16,242539.50000,34745.375000,32438.625000
4,010001,Southeast Health Medical Center,AL,38,EXTRACRANIAL PROCEDURES WITH CC,11,122741.18182,14999.818182,9579.363636


## 5. Compute Hospital-Level Payment Aggregates

### Why we aggregate

The PUF is at the hospital × DRG level, but for our documentation gap analysis we need **hospital-level** payment totals. We aggregate to compute:

- **Total Medicare discharges** per hospital
- **Total Medicare charges** (sum of avg_charges × discharges for each DRG)
- **Total Medicare payments** (sum of avg_payment × discharges for each DRG)
- **Charge-to-payment ratio** (total charges / total payments)
- **Average payment per discharge** (total payments / total discharges)
- **Number of distinct DRGs** billed (a measure of case complexity/diversity)

In [7]:
# ============================================================
# Compute hospital-level aggregates
# ============================================================

# First, compute total charges and payments per row (avg × discharges)
if 'discharges' in df_puf.columns:
    if 'avg_covered_charges' in df_puf.columns:
        df_puf['total_charges'] = df_puf['avg_covered_charges'] * df_puf['discharges']
    if 'avg_total_payment' in df_puf.columns:
        df_puf['total_payments'] = df_puf['avg_total_payment'] * df_puf['discharges']
    if 'avg_medicare_payment' in df_puf.columns:
        df_puf['total_medicare_payments'] = df_puf['avg_medicare_payment'] * df_puf['discharges']

# Aggregate to hospital level
agg_dict = {}
if 'discharges' in df_puf.columns:
    agg_dict['discharges'] = 'sum'
if 'total_charges' in df_puf.columns:
    agg_dict['total_charges'] = 'sum'
if 'total_payments' in df_puf.columns:
    agg_dict['total_payments'] = 'sum'
if 'total_medicare_payments' in df_puf.columns:
    agg_dict['total_medicare_payments'] = 'sum'
if 'drg_code' in df_puf.columns:
    agg_dict['drg_code'] = 'nunique'

# Include hospital name and state if available
group_cols = ['ccn']
name_cols = {}
if 'hospital_name' in df_puf.columns:
    name_cols['hospital_name'] = 'first'
if 'state' in df_puf.columns:
    name_cols['state'] = 'first'

df_hospital_payments = df_puf.groupby('ccn').agg({**agg_dict, **name_cols}).reset_index()

# Rename the DRG count column
if 'drg_code' in df_hospital_payments.columns:
    df_hospital_payments.rename(columns={'drg_code': 'distinct_drgs'}, inplace=True)

# Compute derived metrics
if 'total_charges' in df_hospital_payments.columns and 'total_payments' in df_hospital_payments.columns:
    df_hospital_payments['charge_to_payment_ratio'] = (
        df_hospital_payments['total_charges'] / df_hospital_payments['total_payments']
    )

if 'total_payments' in df_hospital_payments.columns and 'discharges' in df_hospital_payments.columns:
    df_hospital_payments['avg_payment_per_discharge'] = (
        df_hospital_payments['total_payments'] / df_hospital_payments['discharges']
    )

print(f'Hospital-level payment data: {len(df_hospital_payments):,} hospitals')
print(f'\nColumns: {list(df_hospital_payments.columns)}')
df_hospital_payments.head()

Hospital-level payment data: 2,945 hospitals

Columns: ['ccn', 'discharges', 'total_charges', 'total_payments', 'total_medicare_payments', 'distinct_drgs', 'hospital_name', 'state', 'charge_to_payment_ratio', 'avg_payment_per_discharge']


,ccn,discharges,total_charges,total_payments,total_medicare_payments,distinct_drgs,hospital_name,state,charge_to_payment_ratio,avg_payment_per_discharge
0,010001,2972,1.934617e+08,4.275291e+07,3.630956e+07,91,Southeast Health Medical Center,AL,4.525113,14385.230821
1,010005,787,1.637460e+07,7.408886e+06,6.101017e+06,31,Marshall Medical Centers South Campus,AL,2.210129,9414.086404
2,010006,2652,1.483071e+08,3.045992e+07,2.535892e+07,76,North Alabama Medical Center,AL,4.868926,11485.640272
3,010007,126,1.791428e+06,1.304988e+06,1.178734e+06,7,Mizell Memorial Hospital,AL,1.372754,10357.047619
4,010008,11,1.590350e+05,7.307600e+04,5.867600e+04,1,Crenshaw Community Hospital,AL,2.176296,6643.272727


## 6. Analyze DRG-Level Severity Tier Patterns

### Identifying CC/MCC DRG families in the PUF

This is where we connect the PUF data to our documentation gap thesis. For each DRG family that has CC/MCC severity tiers, we can compute:

- What percentage of a hospital's discharges in that family fall in the lowest tier ("without CC")?
- How does this compare across hospitals of similar size/type?

A hospital where 60% of heart failure cases are coded as "without CC" — while its peers average 40% — may be under-documenting comorbidities.

In [8]:
# ============================================================
# Identify DRG severity tiers in the PUF data
# ============================================================
# DRG descriptions typically include severity indicators:
#   "W MCC" or "WITH MCC" = with Major Complication/Comorbidity
#   "W CC" or "WITH CC" = with Complication/Comorbidity  
#   "W/O CC" or "WITHOUT CC" = without CC/MCC

if 'drg_description' in df_puf.columns:
    desc = df_puf['drg_description'].fillna('').str.upper()
    
    # Classify each row by severity tier
    df_puf['severity_tier'] = 'other'
    df_puf.loc[desc.str.contains('W/O CC|WITHOUT CC|W/O MCC', regex=True), 'severity_tier'] = 'without_CC'
    df_puf.loc[
        desc.str.contains('W CC|WITH CC', regex=True) & 
        ~desc.str.contains('W/O CC|WITHOUT CC|MCC', regex=True), 
        'severity_tier'
    ] = 'with_CC'
    df_puf.loc[desc.str.contains('W MCC|WITH MCC', regex=True), 'severity_tier'] = 'with_MCC'
    
    print('Severity tier classification:')
    tier_counts = df_puf['severity_tier'].value_counts()
    for tier, count in tier_counts.items():
        pct = count / len(df_puf) * 100
        print(f'  {tier:15s}: {count:8,} rows ({pct:5.1f}%)')
    
    # How many rows are in CC/MCC DRG families?
    cc_mcc_rows = df_puf[df_puf['severity_tier'] != 'other']
    print(f'\nRows in CC/MCC DRG families: {len(cc_mcc_rows):,} ({len(cc_mcc_rows)/len(df_puf)*100:.1f}%)')
    
    if 'discharges' in df_puf.columns:
        total_discharges = df_puf['discharges'].sum()
        cc_discharges = cc_mcc_rows['discharges'].sum()
        print(f'Discharges in CC/MCC families: {cc_discharges:,.0f} ({cc_discharges/total_discharges*100:.1f}% of total)')
else:
    print('⚠️  No DRG description column to classify severity tiers.')

Severity tier classification:
  with_MCC       :   58,829 rows ( 40.2%)
  other          :   42,590 rows ( 29.1%)
  with_CC        :   36,482 rows ( 24.9%)
  without_CC     :    8,526 rows (  5.8%)

Rows in CC/MCC DRG families: 103,837 (70.9%)
Discharges in CC/MCC families: 3,617,602 (72.9% of total)


In [9]:
# ============================================================
# Compute hospital-level severity tier distribution
# ============================================================
# For each hospital, what % of CC/MCC-family discharges are in each tier?

if 'severity_tier' in df_puf.columns and 'discharges' in df_puf.columns:
    # Filter to only CC/MCC DRG families
    df_cc = df_puf[df_puf['severity_tier'] != 'other'].copy()
    
    # Aggregate discharges by hospital × severity tier
    severity_agg = df_cc.groupby(['ccn', 'severity_tier'])['discharges'].sum().unstack(fill_value=0)
    severity_agg['total_cc_discharges'] = severity_agg.sum(axis=1)
    
    # Compute percentage in each tier
    for tier in ['without_CC', 'with_CC', 'with_MCC']:
        if tier in severity_agg.columns:
            severity_agg[f'pct_{tier}'] = severity_agg[tier] / severity_agg['total_cc_discharges'] * 100
    
    severity_agg = severity_agg.reset_index()
    
    print(f'Severity tier distribution computed for {len(severity_agg):,} hospitals')
    print(f'\nNational average severity tier distribution (weighted by discharges):')
    
    total = severity_agg['total_cc_discharges'].sum()
    for tier in ['without_CC', 'with_CC', 'with_MCC']:
        if tier in severity_agg.columns:
            tier_total = severity_agg[tier].sum()
            print(f'  {tier:15s}: {tier_total/total*100:5.1f}%  ({tier_total:,.0f} discharges)')
    
    print(f'\nHospital-level distribution statistics (% without CC):')
    if 'pct_without_CC' in severity_agg.columns:
        pct_stats = severity_agg['pct_without_CC']
        print(f'  Mean:   {pct_stats.mean():.1f}%')
        print(f'  Median: {pct_stats.median():.1f}%')
        print(f'  Std:    {pct_stats.std():.1f}%')
        print(f'  Min:    {pct_stats.min():.1f}%')
        print(f'  Max:    {pct_stats.max():.1f}%')
        print(f'\nInsight: Hospitals in the top quartile of "% without CC" may have')
        print(f'the most to gain from CDI improvement — their documentation may be')
        print(f'missing comorbidities that would shift cases to higher-paying tiers.')

Severity tier distribution computed for 2,871 hospitals

National average severity tier distribution (weighted by discharges):
  without_CC     :   4.7%  (168,325 discharges)
  with_CC        :  26.2%  (946,276 discharges)
  with_MCC       :  69.2%  (2,503,001 discharges)

Hospital-level distribution statistics (% without CC):
  Mean:   5.3%
  Median: 1.5%
  Std:    14.5%
  Min:    0.0%
  Max:    100.0%

Insight: Hospitals in the top quartile of "% without CC" may have
the most to gain from CDI improvement — their documentation may be
missing comorbidities that would shift cases to higher-paying tiers.


## 7. Payment Distribution Analysis

Let's look at the overall payment landscape to contextualize the revenue gap estimates we'll build in NB08.

In [10]:
# ============================================================
# Hospital-level payment statistics
# ============================================================

print('Hospital-Level Medicare Payment Statistics')
print('=' * 55)

if 'total_payments' in df_hospital_payments.columns:
    payments = df_hospital_payments['total_payments'].dropna()
    print(f'\nTotal Medicare Payments (across all hospitals):')
    print(f'  Sum:    ${payments.sum():,.0f}')
    print(f'  Mean:   ${payments.mean():,.0f} per hospital')
    print(f'  Median: ${payments.median():,.0f} per hospital')

if 'avg_payment_per_discharge' in df_hospital_payments.columns:
    avg_pay = df_hospital_payments['avg_payment_per_discharge'].dropna()
    print(f'\nAverage Payment Per Discharge:')
    print(f'  Mean:   ${avg_pay.mean():,.0f}')
    print(f'  Median: ${avg_pay.median():,.0f}')
    print(f'  Range:  ${avg_pay.min():,.0f} to ${avg_pay.max():,.0f}')

if 'charge_to_payment_ratio' in df_hospital_payments.columns:
    cpr = df_hospital_payments['charge_to_payment_ratio'].dropna()
    print(f'\nCharge-to-Payment Ratio:')
    print(f'  Mean:   {cpr.mean():.2f}x')
    print(f'  Median: {cpr.median():.2f}x')
    print(f'  Range:  {cpr.min():.2f}x to {cpr.max():.2f}x')
    print(f'\n  (A ratio of 3.5x means hospitals charge $3.50 for every $1 Medicare pays.\n'
          f'   Higher ratios may indicate hospitals inflating charges to compensate for\n'
          f'   documentation-driven underpayment on the DRG side.)')

if 'discharges' in df_hospital_payments.columns:
    disch = df_hospital_payments['discharges'].dropna()
    print(f'\nMedicare Discharges Per Hospital:')
    print(f'  Mean:   {disch.mean():,.0f}')
    print(f'  Median: {disch.median():,.0f}')
    print(f'  Total:  {disch.sum():,.0f} Medicare discharges nationally')

Hospital-Level Medicare Payment Statistics

Total Medicare Payments (across all hospitals):
  Sum:    $88,411,099,942
  Mean:   $30,020,747 per hospital
  Median: $11,548,161 per hospital

Average Payment Per Discharge:
  Mean:   $15,589
  Median: $13,532
  Range:  $5,382 to $126,309

Charge-to-Payment Ratio:
  Mean:   4.46x
  Median: 3.89x
  Range:  0.24x to 23.27x

  (A ratio of 3.5x means hospitals charge $3.50 for every $1 Medicare pays.
   Higher ratios may indicate hospitals inflating charges to compensate for
   documentation-driven underpayment on the DRG side.)

Medicare Discharges Per Hospital:
  Mean:   1,684
  Median: 866
  Total:  4,960,325 Medicare discharges nationally


## 8. Save Outputs

We save three files:
1. **Hospital-level payment aggregates** — one row per hospital with total charges, payments, ratios
2. **Hospital severity tier distribution** — one row per hospital with % in each CC/MCC tier
3. **Full DRG-level data** — the detailed hospital × DRG payment data for deep-dives in NB09

In [11]:
# ============================================================
# Save all output files
# ============================================================

# 1. Hospital-level payment aggregates
payment_path = OUTPUT_DIR / 'hospital_payment_data.csv'
df_hospital_payments.to_csv(payment_path, index=False)
print(f'1. Hospital payment data: {len(df_hospital_payments):,} hospitals')
print(f'   → {payment_path}')

# 2. Severity tier distribution
if 'severity_agg' in dir():
    severity_path = OUTPUT_DIR / 'hospital_severity_distribution.csv'
    severity_agg.to_csv(severity_path, index=False)
    print(f'\n2. Severity distribution: {len(severity_agg):,} hospitals')
    print(f'   → {severity_path}')

# 3. Full DRG-level data (for detailed analysis in NB09)
drg_detail_path = OUTPUT_DIR / 'hospital_drg_detail.csv'
df_puf.to_csv(drg_detail_path, index=False)
print(f'\n3. DRG-level detail: {len(df_puf):,} rows')
print(f'   → {drg_detail_path}')

print(f'\n✓ NB03 complete.')
print(f'  → hospital_payment_data.csv joins with NB01 + NB02 via CCN')
print(f'  → Severity distribution feeds into NB07 documentation gap scoring')
print(f'  → DRG detail enables revenue impact modeling in NB08')

1. Hospital payment data: 2,945 hospitals
   → /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb03_payment_data/hospital_payment_data.csv

2. Severity distribution: 2,871 hospitals
   → /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb03_payment_data/hospital_severity_distribution.csv



3. DRG-level detail: 146,427 rows
   → /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb03_payment_data/hospital_drg_detail.csv

✓ NB03 complete.
  → hospital_payment_data.csv joins with NB01 + NB02 via CCN
  → Severity distribution feeds into NB07 documentation gap scoring
  → DRG detail enables revenue impact modeling in NB08


## Appendix: Quick Reference — How These Files Connect

```
NB01 (hospital_characteristics.csv)
 │  Fields: ccn, beds, bed_size_tier, ownership_category, is_teaching, state, peer_group
 │
 ├── JOIN on ccn → NB02 (hospital_drg_data.csv)
 │   Fields: ccn, cmi, discharges, wage_index
 │
 ├── JOIN on ccn → NB03 (hospital_payment_data.csv) 
 │   Fields: ccn, total_payments, charge_to_payment_ratio, avg_payment_per_discharge
 │
 └── JOIN on ccn → NB03 (hospital_severity_distribution.csv)
     Fields: ccn, pct_without_CC, pct_with_CC, pct_with_MCC

All three datasets link via the 6-digit CCN (CMS Certification Number).
NB07 will join them to create the master hospital scoring dataset.
```